# Очистка и подготовка user-course датасета для моделирования статуса

Этот ноутбук собирает рабочую основу для модельного датасета на уровне `user-course`.

Документ устроен так:

- сначала загружаются и типизируются основные таблицы LMS;
- затем проверяются ключи, связность, дубликаты и базовые аномалии;
- после этого строятся агрегаты по сырым логам до уровня `user-course`;
- в конце формируется единая витрина для следующего этапа feature engineering и обучения модели.

Важно: в этом ноутбуке не обрабатываются сырые `stats__module_*`. Вместо них используется уже подготовленная объединённая таблица `./modules/status_modules_complete.csv`, которая задаёт population на уровне модульного `user-course` и содержит фактический статус там, где он уже известен.


In [20]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

# try:
#     from IPython.display import display
# except ImportError:
#     def display(obj):
#         print(obj)

pd.options.display.max_columns = 200
pd.options.display.max_rows = 200


def resolve_base_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "hackathon"]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "modules").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Не удалось найти папки ./src и ./modules относительно текущего ноутбука. "
        "Откройте ноутбук из каталога hackathon или проверьте рабочую директорию."
    )


BASE_DIR = resolve_base_dir()
SRC_DIR = BASE_DIR / "src"
MODULES_DIR = BASE_DIR / "modules"
OUTPUT_DIR = BASE_DIR / "artifacts"
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_KWARGS = {
    "thousands": ",",
    "true_values": ["True"],
    "false_values": ["False"],
}


def read_typed_csv(
    path: Path,
    *,
    usecols: list[str],
    numeric_cols: dict[str, str] | None = None,
    bool_cols: list[str] | None = None,
    category_cols: list[str] | None = None,
    datetime_cols: dict[str, str] | None = None,
    extra_csv_kwargs: dict | None = None,
) -> pd.DataFrame:
    read_kwargs = dict(CSV_KWARGS)
    if extra_csv_kwargs:
        read_kwargs.update(extra_csv_kwargs)

    df = pd.read_csv(path, usecols=usecols, **read_kwargs)

    for col, fmt in (datetime_cols or {}).items():
        df[col] = pd.to_datetime(df[col], format=fmt, errors="coerce")

    for col, dtype in (numeric_cols or {}).items():
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)

    for col in (bool_cols or []):
        df[col] = df[col].astype("boolean")

    for col in (category_cols or []):
        df[col] = df[col].astype("category")

    return df


def frame_memory_mb(df: pd.DataFrame) -> float:
    return round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2)


def bool_sum(series: pd.Series) -> int:
    return int(series.fillna(False).astype("int8").sum())


def pct(numerator: int, denominator: int) -> float:
    return round(numerator / denominator * 100, 2) if denominator else np.nan


def fk_report(name: str, child: pd.Series, parent: pd.Series) -> dict:
    child_non_null = child.dropna()
    parent_index = pd.Index(parent.dropna().unique())
    total = int(child_non_null.shape[0])
    matched = int(child_non_null.isin(parent_index).sum())
    return {
        "check": name,
        "rows_checked": total,
        "matched_rows": matched,
        "unmatched_rows": total - matched,
        "coverage_pct": pct(matched, total),
    }


BASE_DIR


PosixPath('/Users/maria/Desktop/uni/hackathon')

## Что именно загружаем и почему

Здесь читаются только те таблицы, которые реально помогают собрать витрину на уровне `user-course` или нужны для контроля качества связей.

Осознанные решения:

- `stats__module_*` исключены из ноутбука полностью: они обрабатываются отдельно;
- объединённый `modules/status_modules_complete.csv` используется как модульная population-таблица;
- загружаются только те поля, которые нужны для диагностики качества или дальнейших user-course агрегатов;
- таблицы наград здесь не используются: они не course-specific и легко создают риск временной утечки.

Ниже мы сначала подготавливаем typed DataFrame-ы, а затем строим признаки из сырых логов LMS.


In [21]:
users_df = read_typed_csv(
    SRC_DIR / "users.csv",
    usecols=[
        "id",
        "created_at",
        "updated_at",
        "type",
        "sign_in_count",
        "last_sign_in_at",
        "grade_id",
        "subscribed",
        "is_teacher",
        "timezone",
        "grade_changed_at",
        "xp",
        "d_wk_school_id",
        "d_wk_municipal_id",
        "d_wk_region_id",
        "wk_gender",
    ],
    numeric_cols={
        "id": "Int32",
        "sign_in_count": "Int16",
        "grade_id": "Int16",
        "xp": "Int32",
        "d_wk_school_id": "Int32",
        "d_wk_municipal_id": "Int32",
        "d_wk_region_id": "Int32",
        "wk_gender": "Int8",
    },
    bool_cols=["subscribed", "is_teacher"],
    category_cols=["type", "timezone"],
    datetime_cols={
        "created_at": "%d %b, %Y, %H:%M",
        "updated_at": "%d %b, %Y, %H:%M",
        "last_sign_in_at": "%d %b, %Y, %H:%M",
        "grade_changed_at": "%d %b, %Y, %H:%M",
    },
)

users_courses_df = read_typed_csv(
    SRC_DIR / "users_courses.csv",
    usecols=[
        "id",
        "user_id",
        "course_id",
        "state",
        "created_at",
        "updated_at",
        "access_finished_at",
        "wk_points",
        "wk_max_points",
        "wk_max_viewable_lessons",
        "wk_max_task_count",
        "wk_officially_started_at",
        "wk_course_completed_at",
    ],
    numeric_cols={
        "id": "Int32",
        "user_id": "Int32",
        "course_id": "Int32",
        "wk_points": "Float32",
        "wk_max_points": "Float32",
        "wk_max_viewable_lessons": "Float32",
        "wk_max_task_count": "Float32",
    },
    category_cols=["state"],
    datetime_cols={
        "created_at": "%d %b, %Y, %H:%M",
        "updated_at": "%d %b, %Y, %H:%M",
        "access_finished_at": "%d %b, %Y",
        "wk_officially_started_at": "%d %b, %Y",
        "wk_course_completed_at": "%d %b, %Y, %H:%M",
    },
)
users_courses_df = users_courses_df.rename(columns={"id": "users_course_id"})

lessons_df = read_typed_csv(
    SRC_DIR / "lessons.csv",
    usecols=[
        "id",
        "course_id",
        "conspect_expected",
        "task_expected",
        "lesson_number",
        "wk_max_points",
        "wk_task_count",
        "wk_survival_training_expected",
        "wk_scratch_playground_enabled",
        "wk_attendance_tracking_enabled",
        "wk_video_duration",
        "wk_attendance_tracking_disabled_at",
    ],
    numeric_cols={
        "id": "Int32",
        "course_id": "Int32",
        "lesson_number": "Int16",
        "wk_max_points": "Float32",
        "wk_task_count": "Float32",
        "wk_video_duration": "Float32",
    },
    bool_cols=[
        "conspect_expected",
        "task_expected",
        "wk_survival_training_expected",
        "wk_scratch_playground_enabled",
        "wk_attendance_tracking_enabled",
    ],
    datetime_cols={"wk_attendance_tracking_disabled_at": "%d %b, %Y, %H:%M"},
)
lessons_df = lessons_df.rename(columns={"id": "lesson_id"})

groups_df = read_typed_csv(
    SRC_DIR / "groups.csv",
    usecols=[
        "id",
        "lesson_id",
        "teacher_id",
        "starts_at",
        "duration",
        "state",
        "group_template_id",
        "video_available",
        "pupils_notified_at",
        "wk_actual_started_at",
        "wk_actual_finished_at",
        "wk_duration_actual",
    ],
    numeric_cols={
        "id": "Int32",
        "lesson_id": "Int32",
        "teacher_id": "Int32",
        "duration": "Int16",
        "group_template_id": "Int32",
    },
    bool_cols=["video_available", "wk_duration_actual"],
    category_cols=["state"],
    datetime_cols={
        "starts_at": "%d %b, %Y, %H:%M",
        "pupils_notified_at": "%d %b, %Y, %H:%M",
        "wk_actual_started_at": "%d %b, %Y, %H:%M",
        "wk_actual_finished_at": "%d %b, %Y, %H:%M",
    },
)
groups_df = groups_df.rename(columns={"id": "group_id"})

trainings_df = read_typed_csv(
    SRC_DIR / "trainings.csv",
    usecols=["id", "discipline_id", "time_limit", "published_at", "difficulty", "lesson_id", "task_templates_count"],
    numeric_cols={
        "id": "Int32",
        "discipline_id": "Int32",
        "time_limit": "Int16",
        "difficulty": "Int8",
        "lesson_id": "Int32",
        "task_templates_count": "Int16",
    },
    datetime_cols={"published_at": "%d %b, %Y, %H:%M"},
)
trainings_df = trainings_df.rename(columns={"id": "training_id"})

lesson_tasks_df = read_typed_csv(
    SRC_DIR / "lesson_tasks.csv",
    usecols=["id", "lesson_id", "task_id", "position", "task_required"],
    numeric_cols={
        "id": "Int32",
        "lesson_id": "Int32",
        "task_id": "Int32",
        "position": "Int16",
    },
    bool_cols=["task_required"],
)

homeworks_df = read_typed_csv(
    SRC_DIR / "homeworks.csv",
    usecols=["id", "resource_type", "resource_id", "homework_type"],
    numeric_cols={"id": "Int32", "resource_id": "Int32", "homework_type": "Int8"},
    category_cols=["resource_type"],
)
homeworks_df = homeworks_df.rename(columns={"id": "homework_id"})

homework_items_df = read_typed_csv(
    SRC_DIR / "homework_items.csv",
    usecols=["id", "homework_id", "resource_type", "resource_id", "position"],
    numeric_cols={
        "id": "Int32",
        "homework_id": "Int32",
        "resource_id": "Int32",
        "position": "Int16",
    },
    category_cols=["resource_type"],
)

user_access_histories_df = read_typed_csv(
    SRC_DIR / "user_access_histories.csv",
    usecols=["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    numeric_cols={"users_course_id": "Int32"},
    category_cols=["activator_class"],
    datetime_cols={
        "access_started_at": "%d %b, %Y",
        "access_expired_at": "%d %b, %Y",
    },
)

user_lessons_df = read_typed_csv(
    SRC_DIR / "user_lessons.csv",
    usecols=[
        "id",
        "user_id",
        "lesson_id",
        "group_id",
        "video_visited",
        "video_viewed",
        "translation_visited",
        "users_course_id",
        "solved",
        "solved_tasks_count",
        "wk_points",
        "wk_solved_task_count",
    ],
    numeric_cols={
        "id": "Int32",
        "user_id": "Int32",
        "lesson_id": "Int32",
        "group_id": "Int32",
        "users_course_id": "Int32",
        "solved_tasks_count": "Int16",
        "wk_points": "Float32",
        "wk_solved_task_count": "Int16",
    },
    bool_cols=["video_visited", "video_viewed", "translation_visited", "solved"],
)
user_lessons_df = user_lessons_df.rename(columns={"id": "user_lesson_id"})

user_activity_histories_df = read_typed_csv(
    SRC_DIR / "user_activity_histories.csv",
    usecols=["user_lesson_id", "action", "created_at"],
    numeric_cols={"user_lesson_id": "Int32"},
    category_cols=["action"],
    datetime_cols={"created_at": "%Y-%m-%d %H:%M:%S"},
)

user_answers_df = read_typed_csv(
    SRC_DIR / "user_answers.csv",
    usecols=[
        "user_id",
        "task_id",
        "attempts",
        "solved",
        "points",
        "max_attempts",
        "skipped",
        "resource_type",
        "resource_id",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
    numeric_cols={
        "user_id": "Int32",
        "task_id": "Int32",
        "attempts": "Int16",
        "points": "Float32",
        "max_attempts": "Int16",
        "resource_id": "Int32",
        "async_check_status": "Int8",
    },
    bool_cols=["solved", "skipped", "wk_partial_answer"],
    category_cols=["resource_type"],
    datetime_cols={"submitted_at": "%Y-%m-%d %H:%M:%S"},
)

user_trainings_df = read_typed_csv(
    SRC_DIR / "user_trainings.csv",
    usecols=[
        "user_id",
        "training_id",
        "solved_tasks_count",
        "earned_points",
        "type",
        "state",
        "submitted_answers_count",
        "started_at",
        "finished_at",
        "attempts",
        "mark",
        "mark_saved_at",
    ],
    numeric_cols={
        "user_id": "Int32",
        "training_id": "Int32",
        "solved_tasks_count": "Int16",
        "earned_points": "Float32",
        "submitted_answers_count": "Int16",
        "attempts": "Int16",
        "mark": "Float32",
    },
    category_cols=["type", "state"],
    datetime_cols={
        "started_at": "%d %b, %Y, %H:%M",
        "finished_at": "%d %b, %Y, %H:%M",
        "mark_saved_at": "%d %b, %Y, %H:%M",
    },
)

wk_users_courses_actions_df = read_typed_csv(
    SRC_DIR / "wk_users_courses_actions.csv",
    usecols=["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    numeric_cols={"user_id": "Int32", "users_course_id": "Int32", "lesson_id": "Int32"},
    category_cols=["action"],
    datetime_cols={
        "created_at": "%Y-%m-%d %H:%M:%S",
        "updated_at": "%Y-%m-%d %H:%M:%S",
    },
)

wk_media_view_sessions_df = read_typed_csv(
    SRC_DIR / "wk_media_view_sessions.csv",
    usecols=["resource_type", "resource_id", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    numeric_cols={
        "resource_id": "Int32",
        "viewer_id": "Int32",
        "segments_total": "Int16",
        "viewed_segments_count": "Int16",
    },
    category_cols=["resource_type", "kind"],
    datetime_cols={"started_at": "%d %b, %Y, %H:%M"},
)

modules_status_df = read_typed_csv(
    MODULES_DIR / "status_modules_complete.csv",
    usecols=[
        "module",
        "user_id",
        "club_name",
        "teacher_id",
        "course_id",
        "cohort_id",
        "level_bin",
        "enrollment_date",
        "status_actual",
    ],
    numeric_cols={
        "module": "Int8",
        "user_id": "Int32",
        "teacher_id": "Int32",
        "course_id": "Int32",
        "cohort_id": "Int32",
        "level_bin": "Int8",
    },
    category_cols=["club_name", "status_actual"],
    datetime_cols={"enrollment_date": "%Y-%m-%d"},
)

dataframes = {
    "users_df": users_df,
    "users_courses_df": users_courses_df,
    "lessons_df": lessons_df,
    "groups_df": groups_df,
    "trainings_df": trainings_df,
    "lesson_tasks_df": lesson_tasks_df,
    "homeworks_df": homeworks_df,
    "homework_items_df": homework_items_df,
    "user_access_histories_df": user_access_histories_df,
    "user_lessons_df": user_lessons_df,
    "user_activity_histories_df": user_activity_histories_df,
    "user_answers_df": user_answers_df,
    "user_trainings_df": user_trainings_df,
    "wk_users_courses_actions_df": wk_users_courses_actions_df,
    "wk_media_view_sessions_df": wk_media_view_sessions_df,
    "modules_status_df": modules_status_df,
}

inventory_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "rows": [df.shape[0] for df in dataframes.values()],
        "cols": [df.shape[1] for df in dataframes.values()],
        "memory_mb": [frame_memory_mb(df) for df in dataframes.values()],
    }
)

display(inventory_df.sort_values("memory_mb", ascending=False).reset_index(drop=True))
display(modules_status_df.head())
print("Суммарная память по загруженным DataFrame, MB:", round(inventory_df["memory_mb"].sum(), 2))


,dataframe,rows,cols,memory_mb
0,user_answers_df,15176182,12,622.35
1,wk_users_courses_actions_df,12909207,6,393.96
2,user_lessons_df,3070664,12,128.85
3,user_activity_histories_df,3031137,3,40.47
4,user_trainings_df,427628,12,22.43
5,wk_media_view_sessions_df,852358,7,21.14
6,users_courses_df,290835,13,21.08
7,user_access_histories_df,667124,4,14.00
8,users_df,95395,16,6.56
9,groups_df,13076,12,0.75


,module,user_id,club_name,teacher_id,course_id,cohort_id,level_bin,enrollment_date,status_actual
0,1,701741,Python. Начальный уровень. Онлайн. 1 модуль,699869,1029,1149,0,2025-09-19,Завершил
1,1,737977,Python. Начальный уровень. Онлайн. 1 модуль,730341,1029,1216,0,2025-11-05,Завершил
2,1,722851,Python. Базовый уровень. Онлайн. 1 модуль,730208,1030,1186,1,2025-10-21,Отчислен
3,1,709724,Python. Базовый уровень. Онлайн. 1 модуль,700089,1030,1108,1,2025-09-23,Завершил
4,1,717763,Python. Базовый уровень. Онлайн. 1 модуль,718519,1030,1055,1,2025-10-09,Отчислен


Суммарная память по загруженным DataFrame, MB: 1272.76


## Первичная проверка качества

На этом шаге важно не просто посмотреть `shape`, а зафиксировать, что именно можно использовать как надёжную связь для user-course агрегатов.

Проверяем четыре группы вопросов:

- грубые дубликаты и нарушения ключей;
- покрытие FK между таблицами;
- временные аномалии и базовые sanity-checks;
- качество сигналов просмотра, событий внутри урока и стыковку `modules/status_modules_complete.csv` с `users_courses`.


In [22]:
duplicate_rows_df = pd.DataFrame(
    {
        "dataframe": list(dataframes.keys()),
        "duplicate_rows": [int(df.duplicated().sum()) for df in dataframes.values()],
        "rows": [df.shape[0] for df in dataframes.values()],
    }
)
duplicate_rows_df["duplicate_share_pct"] = (
    duplicate_rows_df["duplicate_rows"] / duplicate_rows_df["rows"] * 100
).round(2)

key_uniqueness_df = pd.DataFrame(
    [
        {"table": "users_df", "key": "user_id", "duplicates": int(users_df.duplicated(["id"]).sum())},
        {"table": "users_courses_df", "key": "users_course_id", "duplicates": int(users_courses_df.duplicated(["users_course_id"]).sum())},
        {"table": "users_courses_df", "key": "(user_id, course_id)", "duplicates": int(users_courses_df.duplicated(["user_id", "course_id"]).sum())},
        {"table": "lessons_df", "key": "lesson_id", "duplicates": int(lessons_df.duplicated(["lesson_id"]).sum())},
        {"table": "groups_df", "key": "group_id", "duplicates": int(groups_df.duplicated(["group_id"]).sum())},
        {"table": "trainings_df", "key": "training_id", "duplicates": int(trainings_df.duplicated(["training_id"]).sum())},
        {"table": "lesson_tasks_df", "key": "id", "duplicates": int(lesson_tasks_df.duplicated(["id"]).sum())},
        {"table": "homeworks_df", "key": "homework_id", "duplicates": int(homeworks_df.duplicated(["homework_id"]).sum())},
        {"table": "homework_items_df", "key": "id", "duplicates": int(homework_items_df.duplicated(["id"]).sum())},
        {"table": "user_lessons_df", "key": "user_lesson_id", "duplicates": int(user_lessons_df.duplicated(["user_lesson_id"]).sum())},
        {"table": "user_lessons_df", "key": "(user_id, users_course_id, lesson_id)", "duplicates": int(user_lessons_df.duplicated(["user_id", "users_course_id", "lesson_id"]).sum())},
        {"table": "user_activity_histories_df", "key": "(user_lesson_id, action, created_at)", "duplicates": int(user_activity_histories_df.duplicated(["user_lesson_id", "action", "created_at"]).sum())},
        {"table": "modules_status_df", "key": "(module, user_id, course_id)", "duplicates": int(modules_status_df.duplicated(["module", "user_id", "course_id"]).sum())},
    ]
)

fk_checks_df = pd.DataFrame(
    [
        fk_report("users_courses.user_id -> users.id", users_courses_df["user_id"], users_df["id"]),
        fk_report("user_lessons.user_id -> users.id", user_lessons_df["user_id"], users_df["id"]),
        fk_report("user_lessons.users_course_id -> users_courses.users_course_id", user_lessons_df["users_course_id"], users_courses_df["users_course_id"]),
        fk_report("user_lessons.lesson_id -> lessons.lesson_id", user_lessons_df["lesson_id"], lessons_df["lesson_id"]),
        fk_report("user_lessons.group_id -> groups.group_id", user_lessons_df["group_id"], groups_df["group_id"]),
        fk_report("user_activity_histories.user_lesson_id -> user_lessons.user_lesson_id", user_activity_histories_df["user_lesson_id"], user_lessons_df["user_lesson_id"]),
        fk_report("groups.lesson_id -> lessons.lesson_id", groups_df["lesson_id"], lessons_df["lesson_id"]),
        fk_report("trainings.lesson_id -> lessons.lesson_id", trainings_df["lesson_id"], lessons_df["lesson_id"]),
        fk_report("user_trainings.training_id -> trainings.training_id", user_trainings_df["training_id"], trainings_df["training_id"]),
        fk_report("wk_media_view_sessions.viewer_id -> users.id", wk_media_view_sessions_df["viewer_id"], users_df["id"]),
        fk_report(
            "wk_media_view_sessions.resource_id[Group] -> groups.group_id",
            wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Group"), "resource_id"],
            groups_df["group_id"],
        ),
        fk_report(
            "wk_media_view_sessions.resource_id[Lesson] -> lessons.lesson_id",
            wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Lesson"), "resource_id"],
            lessons_df["lesson_id"],
        ),
        fk_report("user_answers.user_id -> users.id", user_answers_df["user_id"], users_df["id"]),
        fk_report("modules_status.user_id -> users.id", modules_status_df["user_id"], users_df["id"]),
        fk_report("modules_status.course_id -> lessons.course_id", modules_status_df["course_id"], lessons_df["course_id"]),
        fk_report("modules_status.cohort_id -> groups.group_template_id", modules_status_df["cohort_id"], groups_df["group_template_id"]),
    ]
)

temporal_checks_df = pd.DataFrame(
    [
        {"table": "users_df", "check": "updated_at < created_at", "bad_rows": int((users_df["updated_at"] < users_df["created_at"]).sum())},
        {"table": "users_df", "check": "grade_changed_at < created_at", "bad_rows": int((users_df["grade_changed_at"] < users_df["created_at"]).sum())},
        {"table": "users_courses_df", "check": "updated_at < created_at", "bad_rows": int((users_courses_df["updated_at"] < users_courses_df["created_at"]).sum())},
        {"table": "users_courses_df", "check": "wk_points > wk_max_points", "bad_rows": int((users_courses_df["wk_points"] > users_courses_df["wk_max_points"]).sum())},
        {"table": "groups_df", "check": "wk_actual_finished_at < wk_actual_started_at", "bad_rows": int((groups_df["wk_actual_finished_at"] < groups_df["wk_actual_started_at"]).sum())},
        {"table": "lessons_df", "check": "lesson_number <= 0", "bad_rows": int((lessons_df["lesson_number"] <= 0).sum())},
        {"table": "lesson_tasks_df", "check": "position <= 0", "bad_rows": int((lesson_tasks_df["position"] <= 0).sum())},
        {"table": "homework_items_df", "check": "position <= 0", "bad_rows": int((homework_items_df["position"] <= 0).sum())},
        {"table": "user_answers_df", "check": "attempts > max_attempts", "bad_rows": int((user_answers_df["attempts"] > user_answers_df["max_attempts"]).sum())},
        {"table": "user_trainings_df", "check": "finished_at < started_at", "bad_rows": int((user_trainings_df["finished_at"] < user_trainings_df["started_at"]).sum())},
        {"table": "wk_users_courses_actions_df", "check": "updated_at < created_at", "bad_rows": int((wk_users_courses_actions_df["updated_at"] < wk_users_courses_actions_df["created_at"]).sum())},
        {"table": "wk_media_view_sessions_df", "check": "viewed_segments_count > segments_total", "bad_rows": int((wk_media_view_sessions_df["viewed_segments_count"] > wk_media_view_sessions_df["segments_total"]).sum())},
    ]
)

video_signal_df = pd.DataFrame(
    [
        {"metric": "rows_video_visited_true", "value": int(user_lessons_df["video_visited"].fillna(False).sum())},
        {"metric": "rows_video_viewed_true", "value": int(user_lessons_df["video_viewed"].fillna(False).sum())},
        {"metric": "video_visited_true_and_video_viewed_false", "value": int(((user_lessons_df["video_visited"] == True) & (user_lessons_df["video_viewed"] == False)).sum())},
        {"metric": "video_viewed_true_and_video_visited_false", "value": int(((user_lessons_df["video_viewed"] == True) & (user_lessons_df["video_visited"] == False)).sum())},
    ]
)

activity_probe_df = (
    user_activity_histories_df
    .merge(user_lessons_df[["user_lesson_id", "user_id", "lesson_id", "users_course_id"]], on="user_lesson_id", how="left")
    .merge(lessons_df[["lesson_id", "course_id"]], on="lesson_id", how="left")
)
activity_quality_df = pd.DataFrame(
    [
        {"metric": "user_activity rows with matched user_lesson", "value": int(activity_probe_df["user_id"].notna().sum())},
        {"metric": "user_activity rows with matched course_id", "value": int(activity_probe_df["course_id"].notna().sum())},
        {"metric": "user_lessons rows with any activity", "value": int(user_lessons_df["user_lesson_id"].isin(user_activity_histories_df["user_lesson_id"]).sum())},
        {"metric": "user_activity distinct actions", "value": int(user_activity_histories_df["action"].nunique(dropna=True))},
    ]
)
activity_actions_df = (
    user_activity_histories_df["action"]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("action")
    .reset_index(name="rows")
)

modules_backbone_df = modules_status_df.merge(
    users_courses_df[["users_course_id", "user_id", "course_id"]],
    on=["user_id", "course_id"],
    how="left",
)
module_users_course_coverage_df = (
    modules_backbone_df.assign(has_users_course_id=modules_backbone_df["users_course_id"].notna())
    .groupby("module", dropna=False)
    .agg(
        rows=("module", "size"),
        matched_users_course=("users_course_id", lambda s: int(s.notna().sum())),
        unmatched_users_course=("users_course_id", lambda s: int(s.isna().sum())),
    )
    .reset_index()
)
module_users_course_coverage_df["matched_pct"] = module_users_course_coverage_df.apply(
    lambda row: pct(int(row["matched_users_course"]), int(row["rows"])),
    axis=1,
)

display(duplicate_rows_df.sort_values("duplicate_rows", ascending=False).reset_index(drop=True))
display(key_uniqueness_df)
display(fk_checks_df.sort_values("coverage_pct", ascending=True).reset_index(drop=True))
display(temporal_checks_df.sort_values(["bad_rows", "table"], ascending=[False, True]).reset_index(drop=True))
display(video_signal_df)
display(activity_quality_df)
display(activity_actions_df)
display(module_users_course_coverage_df)


,dataframe,duplicate_rows,rows,duplicate_share_pct
0,wk_users_courses_actions_df,5491157,12909207,42.54
1,user_answers_df,4950084,15176182,32.62
2,user_access_histories_df,355372,667124,53.27
3,user_activity_histories_df,33312,3031137,1.10
4,wk_media_view_sessions_df,3558,852358,0.42
5,users_df,0,95395,0.00
6,users_courses_df,0,290835,0.00
7,lessons_df,0,3369,0.00
8,groups_df,0,13076,0.00
9,trainings_df,0,410,0.00


,table,key,duplicates
0,users_df,user_id,0
1,users_courses_df,users_course_id,0
2,users_courses_df,"(user_id, course_id)",0
3,lessons_df,lesson_id,0
4,groups_df,group_id,0
5,trainings_df,training_id,0
6,lesson_tasks_df,id,0
7,homeworks_df,homework_id,0
8,homework_items_df,id,0
9,user_lessons_df,user_lesson_id,0


,check,rows_checked,matched_rows,unmatched_rows,coverage_pct
0,trainings.lesson_id -> lessons.lesson_id,256,252,4,98.44
1,user_activity_histories.user_lesson_id -> user...,3031137,3007406,23731,99.22
2,user_lessons.lesson_id -> lessons.lesson_id,3070664,3062469,8195,99.73
3,user_lessons.group_id -> groups.group_id,3070664,3062469,8195,99.73
4,groups.lesson_id -> lessons.lesson_id,13076,13068,8,99.94
5,users_courses.user_id -> users.id,290835,290835,0,100.00
6,user_lessons.user_id -> users.id,3070664,3070615,49,100.00
7,user_lessons.users_course_id -> users_courses....,3070664,3070524,140,100.00
8,user_trainings.training_id -> trainings.traini...,427628,427628,0,100.00
9,wk_media_view_sessions.viewer_id -> users.id,852358,852358,0,100.00


,table,check,bad_rows
0,user_answers_df,attempts > max_attempts,190
1,users_courses_df,wk_points > wk_max_points,179
2,wk_users_courses_actions_df,updated_at < created_at,1
3,groups_df,wk_actual_finished_at < wk_actual_started_at,0
4,homework_items_df,position <= 0,0
5,lesson_tasks_df,position <= 0,0
6,lessons_df,lesson_number <= 0,0
7,user_trainings_df,finished_at < started_at,0
8,users_courses_df,updated_at < created_at,0
9,users_df,updated_at < created_at,0


,metric,value
0,rows_video_visited_true,2471536
1,rows_video_viewed_true,99650
2,video_visited_true_and_video_viewed_false,2371889
3,video_viewed_true_and_video_visited_false,3


,metric,value
0,user_activity rows with matched user_lesson,3007406
1,user_activity rows with matched course_id,2999164
2,user_lessons rows with any activity,2525447
3,user_activity distinct actions,3


,action,rows
0,visit_video,2642950
1,show_conspect,278340
2,visit_translation,109847


,module,rows,matched_users_course,unmatched_users_course,matched_pct
0,1,2972,2972,0,100.0
1,2,1955,1955,0,100.0
2,3,1785,1785,0,100.0
3,4,1689,0,1689,0.0


## Интерпретация промежуточных результатов

Здесь важно зафиксировать несколько практических выводов, которые будут влиять на итоговую витрину.

1. `video_viewed` действительно нужен.
   В текущих данных огромное число строк имеет `video_visited=True`, но `video_viewed=False`. Это означает, что одного признака захода в видео недостаточно: для модели важнее отделять факт открытия плеера от фактического просмотра.

2. `user_activity_histories` можно связать с `user_lessons` по `user_lesson_id`.
   Это позволяет использовать события внутри урока не только для ручной проверки, но и для построения user-course агрегатов. При этом покрытие по `course_id` всё равно чуть ниже 100%, потому что часть `user_lessons` не стыкуется с `lessons`, и это ограничение надо помнить при интерпретации признаков.

3. Для модулей 1, 2 и 3 записи из `modules/status_modules_complete.csv` хорошо стыкуются с `users_courses`, а для модуля 4 такого совпадения нет.
   Поэтому итоговую витрину нельзя строить только через `users_course_id`.
   В этом документе используется смешанный подход: где связь с `users_courses` есть, берём её напрямую; где её нет, восстанавливаем принадлежность к курсу через уроки, группы, тренинги и типы ресурсов в логах.


## Удаление точных дублей перед сборкой витрины

На этом шаге удаляются только точные дубли событий в тех таблицах, где они завышают агрегаты.

Важно: здесь не выполняется агрессивная дедупликация по упрощённым ключам вроде `(user_id, task_id)` или `(user_id, action)`. Удаляются только строки, совпадающие по полной сигнатуре события, чтобы не потерять реальные повторные действия пользователя.


In [23]:
def drop_exact_duplicates(df: pd.DataFrame, subset: list[str], dataframe_name: str) -> tuple[pd.DataFrame, dict]:
    duplicate_rows = int(df.duplicated(subset=subset).sum())
    deduped_df = df.drop_duplicates(subset=subset).copy()
    return deduped_df, {
        "dataframe": dataframe_name,
        "rows_before": int(df.shape[0]),
        "duplicate_rows_removed": duplicate_rows,
        "rows_after": int(deduped_df.shape[0]),
        "dedup_subset": ", ".join(subset),
    }


user_activity_histories_df, user_activity_dedup_report = drop_exact_duplicates(
    user_activity_histories_df,
    ["user_lesson_id", "action", "created_at"],
    "user_activity_histories_df",
)
wk_users_courses_actions_df, wk_users_courses_actions_dedup_report = drop_exact_duplicates(
    wk_users_courses_actions_df,
    ["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    "wk_users_courses_actions_df",
)
wk_media_view_sessions_df, wk_media_view_sessions_dedup_report = drop_exact_duplicates(
    wk_media_view_sessions_df,
    ["resource_type", "resource_id", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    "wk_media_view_sessions_df",
)
user_answers_df, user_answers_dedup_report = drop_exact_duplicates(
    user_answers_df,
    [
        "user_id",
        "task_id",
        "attempts",
        "solved",
        "points",
        "max_attempts",
        "skipped",
        "resource_type",
        "resource_id",
        "submitted_at",
        "wk_partial_answer",
        "async_check_status",
    ],
    "user_answers_df",
)
user_access_histories_df, user_access_histories_dedup_report = drop_exact_duplicates(
    user_access_histories_df,
    ["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    "user_access_histories_df",
)

deduplication_df = pd.DataFrame(
    [
        user_activity_dedup_report,
        wk_users_courses_actions_dedup_report,
        wk_media_view_sessions_dedup_report,
        user_answers_dedup_report,
        user_access_histories_dedup_report,
    ]
)
deduplication_df["removed_share_pct"] = deduplication_df.apply(
    lambda row: pct(int(row["duplicate_rows_removed"]), int(row["rows_before"])),
    axis=1,
)

dataframes.update(
    {
        "user_activity_histories_df": user_activity_histories_df,
        "wk_users_courses_actions_df": wk_users_courses_actions_df,
        "wk_media_view_sessions_df": wk_media_view_sessions_df,
        "user_answers_df": user_answers_df,
        "user_access_histories_df": user_access_histories_df,
    }
)

display(deduplication_df)


,dataframe,rows_before,duplicate_rows_removed,rows_after,dedup_subset,removed_share_pct
0,user_activity_histories_df,3031137,33312,2997825,"user_lesson_id, action, created_at",1.10
1,wk_users_courses_actions_df,12909207,5491157,7418050,"user_id, users_course_id, action, created_at, ...",42.54
2,wk_media_view_sessions_df,852358,3558,848800,"resource_type, resource_id, viewer_id, segment...",0.42
3,user_answers_df,15176182,4950084,10226098,"user_id, task_id, attempts, solved, points, ma...",32.62
4,user_access_histories_df,667124,355372,311752,"users_course_id, access_started_at, access_exp...",53.27


## Сборка общей витрины и проверка ее качества

In [24]:
lesson_course_df = lessons_df[["lesson_id", "course_id", "lesson_number", "wk_video_duration"]].copy()

group_course_df = groups_df[["group_id", "lesson_id", "group_template_id", "starts_at"]].merge(
    lesson_course_df[["lesson_id", "course_id"]],
    on="lesson_id",
    how="left",
)

training_course_df = trainings_df[["training_id", "lesson_id", "task_templates_count"]].merge(
    lesson_course_df[["lesson_id", "course_id"]],
    on="lesson_id",
    how="left",
)

homework_course_df = (
    homeworks_df.loc[homeworks_df["resource_type"].eq("Lesson"), ["homework_id", "resource_id"]]
    .rename(columns={"resource_id": "lesson_id"})
    .merge(lesson_course_df[["lesson_id", "course_id"]], on="lesson_id", how="left")
)

homework_item_course_df = homework_items_df[["homework_id", "resource_type", "resource_id", "position"]].merge(
    homework_course_df[["homework_id", "course_id"]],
    on="homework_id",
    how="left",
)

course_mapping_inventory_df = pd.DataFrame(
    [
        {"mapping": "groups -> course_id через lessons", "rows": int(group_course_df.shape[0]), "mapped_rows": int(group_course_df["course_id"].notna().sum())},
        {"mapping": "trainings -> course_id через lessons", "rows": int(training_course_df.shape[0]), "mapped_rows": int(training_course_df["course_id"].notna().sum())},
        {"mapping": "homeworks[Lesson] -> course_id через lessons", "rows": int(homework_course_df.shape[0]), "mapped_rows": int(homework_course_df["course_id"].notna().sum())},
        {"mapping": "homework_items -> course_id через homework_id", "rows": int(homework_item_course_df.shape[0]), "mapped_rows": int(homework_item_course_df["course_id"].notna().sum())},
    ]
)
course_mapping_inventory_df["coverage_pct"] = course_mapping_inventory_df.apply(
    lambda row: pct(int(row["mapped_rows"]), int(row["rows"])),
    axis=1,
)

course_structure_df = (
    lessons_df.groupby("course_id", observed=True)
    .agg(
        course_lessons_total=("lesson_id", "nunique"),
        course_conspect_expected_count=("conspect_expected", bool_sum),
        course_task_expected_count=("task_expected", bool_sum),
        course_survival_training_expected_count=("wk_survival_training_expected", bool_sum),
        course_attendance_tracking_enabled_count=("wk_attendance_tracking_enabled", bool_sum),
        course_video_duration_total=("wk_video_duration", "sum"),
        course_video_duration_mean=("wk_video_duration", "mean"),
    )
    .reset_index()
)

course_tasks_df = (
    lesson_tasks_df.merge(lesson_course_df[["lesson_id", "course_id"]], on="lesson_id", how="left")
    .groupby("course_id", dropna=False, observed=True)
    .agg(
        course_lesson_tasks_total=("task_id", "size"),
        course_lesson_tasks_unique=("task_id", "nunique"),
        course_required_tasks_total=("task_required", bool_sum),
    )
    .reset_index()
)

course_trainings_df = (
    training_course_df.groupby("course_id", dropna=False, observed=True)
    .agg(
        course_trainings_total=("training_id", "nunique"),
        course_training_tasks_total=("task_templates_count", "sum"),
    )
    .reset_index()
)

course_homeworks_df = (
    homework_course_df.groupby("course_id", dropna=False, observed=True)
    .agg(course_homeworks_total=("homework_id", "nunique"))
    .reset_index()
)

course_homework_items_df = (
    homework_item_course_df.groupby("course_id", dropna=False, observed=True)
    .agg(course_homework_items_total=("resource_id", "size"))
    .reset_index()
)

course_features_df = (
    course_structure_df
    .merge(course_tasks_df, on="course_id", how="left")
    .merge(course_trainings_df, on="course_id", how="left")
    .merge(course_homeworks_df, on="course_id", how="left")
    .merge(course_homework_items_df, on="course_id", how="left")
)

user_profile_df = users_df.rename(
    columns={
        "id": "user_id",
        "type": "user_type",
        "created_at": "user_created_at",
        "updated_at": "user_updated_at",
        "last_sign_in_at": "user_last_sign_in_at",
        "grade_changed_at": "user_grade_changed_at",
    }
)

users_courses_feature_df = users_courses_df.copy()
users_courses_feature_df["course_points_ratio"] = (
    users_courses_feature_df["wk_points"]
    / users_courses_feature_df["wk_max_points"].replace({0: np.nan})
)

access_course_df = user_access_histories_df.merge(
    users_courses_feature_df[["users_course_id", "user_id", "course_id"]],
    on="users_course_id",
    how="left",
)
access_course_df["access_days"] = (
    access_course_df["access_expired_at"] - access_course_df["access_started_at"]
).dt.days
access_agg_df = (
    access_course_df.groupby(["user_id", "course_id"], observed=True)
    .agg(
        access_periods_count=("users_course_id", "size"),
        access_days_total=("access_days", "sum"),
        access_started_at_first=("access_started_at", "min"),
        access_expired_at_last=("access_expired_at", "max"),
    )
    .reset_index()
)

user_lessons_course_df = user_lessons_df.merge(
    lesson_course_df[["lesson_id", "course_id"]],
    on="lesson_id",
    how="left",
)
user_lessons_agg_df = (
    user_lessons_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        user_lessons_rows=("lesson_id", "size"),
        user_lessons_nunique=("user_lesson_id", "nunique"),
        lessons_logged_nunique=("lesson_id", "nunique"),
        groups_logged_nunique=("group_id", "nunique"),
        video_visited_lessons=("video_visited", bool_sum),
        video_viewed_lessons=("video_viewed", bool_sum),
        translation_visited_lessons=("translation_visited", bool_sum),
        solved_lessons_count=("solved", bool_sum),
        solved_tasks_total=("solved_tasks_count", "sum"),
        user_lessons_points_sum=("wk_points", "sum"),
        wk_solved_task_count_total=("wk_solved_task_count", "sum"),
    )
    .reset_index()
)
user_lessons_agg_df["video_viewed_share_by_logged_lessons"] = (
    user_lessons_agg_df["video_viewed_lessons"]
    / user_lessons_agg_df["lessons_logged_nunique"].replace({0: np.nan})
)

user_activity_course_df = user_activity_histories_df.merge(
    user_lessons_course_df[["user_lesson_id", "user_id", "lesson_id", "users_course_id", "course_id"]],
    on="user_lesson_id",
    how="left",
)
user_activity_agg_df = (
    user_activity_course_df.loc[user_activity_course_df["course_id"].notna()]
    .groupby(["user_id", "course_id"], observed=True)
    .agg(
        lesson_activity_events_total=("action", "size"),
        lesson_activity_user_lessons_nunique=("user_lesson_id", "nunique"),
        lesson_activity_days=("created_at", lambda s: s.dt.normalize().nunique()),
        lesson_activity_first_at=("created_at", "min"),
        lesson_activity_last_at=("created_at", "max"),
    )
    .reset_index()
)
activity_counts_wide_df = (
    user_activity_course_df.loc[user_activity_course_df["course_id"].notna(), ["user_id", "course_id", "action"]]
    .assign(event_count=1)
    .pivot_table(
        index=["user_id", "course_id"],
        columns="action",
        values="event_count",
        aggfunc="sum",
        fill_value=0,
    )
    .rename_axis(columns=None)
    .add_prefix("lesson_activity_count__")
    .reset_index()
)
user_activity_agg_df = user_activity_agg_df.merge(activity_counts_wide_df, on=["user_id", "course_id"], how="left")

actions_course_df = wk_users_courses_actions_df.merge(
    users_courses_feature_df[["users_course_id", "user_id", "course_id"]].rename(columns={"user_id": "user_id_from_uc"}),
    on="users_course_id",
    how="left",
)
actions_agg_df = (
    actions_course_df.loc[actions_course_df["course_id"].notna()]
    .groupby(["user_id_from_uc", "course_id"], observed=True)
    .agg(
        actions_total=("action", "size"),
        action_days=("created_at", lambda s: s.dt.normalize().nunique()),
        actions_with_lesson_id=("lesson_id", lambda s: int(s.notna().sum())),
        action_first_at=("created_at", "min"),
        action_last_at=("created_at", "max"),
    )
    .reset_index()
    .rename(columns={"user_id_from_uc": "user_id"})
)
action_counts_wide_df = (
    actions_course_df.loc[actions_course_df["course_id"].notna(), ["user_id_from_uc", "course_id", "action"]]
    .assign(event_count=1)
    .pivot_table(
        index=["user_id_from_uc", "course_id"],
        columns="action",
        values="event_count",
        aggfunc="sum",
        fill_value=0,
    )
    .rename_axis(columns=None)
    .add_prefix("action_count__")
    .reset_index()
    .rename(columns={"user_id_from_uc": "user_id"})
)
actions_agg_df = actions_agg_df.merge(action_counts_wide_df, on=["user_id", "course_id"], how="left")

media_group_df = (
    wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Group")]
    .merge(group_course_df[["group_id", "course_id"]], left_on="resource_id", right_on="group_id", how="left")
    .rename(columns={"viewer_id": "user_id"})
)
media_lesson_df = (
    wk_media_view_sessions_df.loc[wk_media_view_sessions_df["resource_type"].eq("Lesson")]
    .merge(lesson_course_df[["lesson_id", "course_id"]], left_on="resource_id", right_on="lesson_id", how="left")
    .rename(columns={"viewer_id": "user_id"})
)
media_course_df = pd.concat([media_group_df, media_lesson_df], ignore_index=True, sort=False)
media_course_df["watch_share"] = np.where(
    media_course_df["segments_total"].gt(0),
    media_course_df["viewed_segments_count"] / media_course_df["segments_total"],
    np.nan,
)
media_agg_df = (
    media_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        media_sessions_total=("resource_id", "size"),
        media_resources_nunique=("resource_id", "nunique"),
        media_watch_share_mean=("watch_share", "mean"),
        media_watch_share_max=("watch_share", "max"),
        media_sessions_80pct=("watch_share", lambda s: int((s >= 0.8).sum())),
        media_sessions_full=("watch_share", lambda s: int((s >= 1.0).sum())),
        media_started_at_first=("started_at", "min"),
        media_started_at_last=("started_at", "max"),
    )
    .reset_index()
)

user_trainings_course_df = user_trainings_df.merge(
    training_course_df[["training_id", "course_id"]],
    on="training_id",
    how="left",
)
user_trainings_agg_df = (
    user_trainings_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        trainings_rows=("training_id", "size"),
        trainings_unique_total=("training_id", "nunique"),
        training_solved_tasks_total=("solved_tasks_count", "sum"),
        training_earned_points_sum=("earned_points", "sum"),
        training_submitted_answers_sum=("submitted_answers_count", "sum"),
        training_attempts_sum=("attempts", "sum"),
        training_mark_max=("mark", "max"),
        training_started_at_first=("started_at", "min"),
        training_finished_at_last=("finished_at", "max"),
    )
    .reset_index()
)

answers_base_cols = [
    "user_id",
    "task_id",
    "attempts",
    "solved",
    "points",
    "max_attempts",
    "skipped",
    "resource_id",
    "submitted_at",
    "wk_partial_answer",
    "async_check_status",
]
answers_lesson_df = (
    user_answers_df.loc[user_answers_df["resource_type"].eq("Lesson"), answers_base_cols]
    .merge(lesson_course_df[["lesson_id", "course_id"]], left_on="resource_id", right_on="lesson_id", how="left")
)
answers_training_df = (
    user_answers_df.loc[user_answers_df["resource_type"].eq("Training"), answers_base_cols]
    .merge(training_course_df[["training_id", "course_id"]], left_on="resource_id", right_on="training_id", how="left")
)
answers_homework_df = (
    user_answers_df.loc[user_answers_df["resource_type"].eq("Homework"), answers_base_cols]
    .merge(homework_course_df[["homework_id", "course_id"]], left_on="resource_id", right_on="homework_id", how="left")
)
user_answers_course_df = pd.concat(
    [answers_lesson_df, answers_training_df, answers_homework_df],
    ignore_index=True,
    sort=False,
)
answer_mapping_coverage_df = pd.DataFrame(
    [
        {"resource_type": "Lesson", "rows": int(answers_lesson_df.shape[0]), "mapped_rows": int(answers_lesson_df["course_id"].notna().sum())},
        {"resource_type": "Training", "rows": int(answers_training_df.shape[0]), "mapped_rows": int(answers_training_df["course_id"].notna().sum())},
        {"resource_type": "Homework", "rows": int(answers_homework_df.shape[0]), "mapped_rows": int(answers_homework_df["course_id"].notna().sum())},
    ]
)
answer_mapping_coverage_df["coverage_pct"] = answer_mapping_coverage_df.apply(
    lambda row: pct(int(row["mapped_rows"]), int(row["rows"])),
    axis=1,
)
user_answers_agg_df = (
    user_answers_course_df.groupby(["user_id", "course_id"], dropna=False, observed=True)
    .agg(
        answer_rows=("task_id", "size"),
        answer_tasks_nunique=("task_id", "nunique"),
        answer_solved_count=("solved", bool_sum),
        answer_skipped_count=("skipped", bool_sum),
        answer_partial_count=("wk_partial_answer", bool_sum),
        answer_points_sum=("points", "sum"),
        answer_attempts_sum=("attempts", "sum"),
        answer_max_attempts_sum=("max_attempts", "sum"),
        answer_async_nonzero_count=("async_check_status", lambda s: int((s.fillna(0) != 0).sum())),
        answer_first_submitted_at=("submitted_at", "min"),
        answer_last_submitted_at=("submitted_at", "max"),
    )
    .reset_index()
)

modeling_population_df = modules_status_df[
    ["module", "user_id", "course_id", "club_name", "teacher_id", "cohort_id", "level_bin", "enrollment_date", "status_actual"]
].copy()
modeling_population_df["target"] = modeling_population_df["status_actual"].map({"Отчислен": 0, "Завершил": 1}).astype("Int8")

modeling_dataset_df = (
    modeling_population_df
    .merge(users_courses_feature_df, on=["user_id", "course_id"], how="left")
    .merge(user_profile_df, on="user_id", how="left")
    .merge(course_features_df, on="course_id", how="left")
    .merge(access_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_lessons_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_activity_agg_df, on=["user_id", "course_id"], how="left")
    .merge(actions_agg_df, on=["user_id", "course_id"], how="left")
    .merge(media_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_trainings_agg_df, on=["user_id", "course_id"], how="left")
    .merge(user_answers_agg_df, on=["user_id", "course_id"], how="left")
)

coverage_by_module_df = (
    modeling_dataset_df.groupby("module", dropna=False)
    .agg(
        rows=("module", "size"),
        labeled_rows=("target", lambda s: int(s.notna().sum())),
        users_course_rows=("users_course_id", lambda s: int(s.notna().sum())),
        access_feature_rows=("access_periods_count", lambda s: int(s.notna().sum())),
        user_lessons_feature_rows=("user_lessons_rows", lambda s: int(s.notna().sum())),
        user_activity_feature_rows=("lesson_activity_events_total", lambda s: int(s.notna().sum())),
        actions_feature_rows=("actions_total", lambda s: int(s.notna().sum())),
        media_feature_rows=("media_sessions_total", lambda s: int(s.notna().sum())),
        trainings_feature_rows=("trainings_rows", lambda s: int(s.notna().sum())),
        answers_feature_rows=("answer_rows", lambda s: int(s.notna().sum())),
    )
    .reset_index()
)
for col in [
    "labeled_rows",
    "users_course_rows",
    "access_feature_rows",
    "user_lessons_feature_rows",
    "user_activity_feature_rows",
    "actions_feature_rows",
    "media_feature_rows",
    "trainings_feature_rows",
    "answers_feature_rows",
]:
    coverage_by_module_df[col.replace("_rows", "_pct")] = coverage_by_module_df.apply(
        lambda row, src=col: pct(int(row[src]), int(row["rows"])),
        axis=1,
    )

missingness_base_df = modeling_dataset_df.loc[modeling_dataset_df["module"] != 4].copy()

missingness_df = (
    missingness_base_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("na_share")
    .reset_index()
    .rename(columns={"index": "column"})
)
missingness_df["na_pct"] = (missingness_df["na_share"] * 100).round(2)

export_path = OUTPUT_DIR / "user_course_modeling_base.csv"
modeling_dataset_df.to_csv(export_path, index=False)

print("Для какой доли строк удалось восстановить course_id?")
display(course_mapping_inventory_df)
display(answer_mapping_coverage_df)

print("Таблица покрытия итоговой user-course витрины по модулям:")
print("какая часть user-courses из stats_module таблиц найдена в инальной табилце из логов лмс")
display(coverage_by_module_df)
print("Таблица пропусков по итоговой витрине без module == 4.")
display(missingness_df.head(40))



/var/folders/7n/464mds813q94_cwkxlkbc_z40000gn/T/ipykernel_12283/3227064302.py:175: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  user_activity_course_df.loc[user_activity_course_df["course_id"].notna(), ["user_id", "course_id", "action"]]
/var/folders/7n/464mds813q94_cwkxlkbc_z40000gn/T/ipykernel_12283/3227064302.py:209: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  actions_course_df.loc[actions_course_df["course_id"].notna(), ["user_id_from_uc", "course_id", "action"]]


Для какой доли строк удалось восстановить course_id?


,mapping,rows,mapped_rows,coverage_pct
0,groups -> course_id через lessons,13076,13068,99.94
1,trainings -> course_id через lessons,410,252,61.46
2,homeworks[Lesson] -> course_id через lessons,1207,332,27.51
3,homework_items -> course_id через homework_id,5901,997,16.90


,resource_type,rows,mapped_rows,coverage_pct
0,Lesson,6669122,6657186,99.82
1,Training,3263364,3237117,99.20
2,Homework,293612,293610,100.00


Таблица покрытия итоговой user-course витрины по модулям:
какая часть user-courses из stats_module таблиц найдена в инальной табилце из логов лмс


,module,rows,labeled_rows,users_course_rows,access_feature_rows,user_lessons_feature_rows,user_activity_feature_rows,actions_feature_rows,media_feature_rows,trainings_feature_rows,answers_feature_rows,labeled_pct,users_course_pct,access_feature_pct,user_lessons_feature_pct,user_activity_feature_pct,actions_feature_pct,media_feature_pct,trainings_feature_pct,answers_feature_pct
0,1,2972,2972,2972,2972,2494,2472,2486,2415,2116,2393,100.0,100.0,100.0,83.92,83.18,83.65,81.26,71.20,80.52
1,2,1955,1955,1955,1955,1915,1913,1913,1908,1846,1870,100.0,100.0,100.0,97.95,97.85,97.85,97.60,94.42,95.65
2,3,1785,0,1785,1785,1770,1768,1774,1748,1321,1744,0.0,100.0,100.0,99.16,99.05,99.38,97.93,74.01,97.70
3,4,1689,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00


Таблица пропусков по итоговой витрине без module == 4.


,column,na_share,na_pct
0,course_video_duration_mean,1.000000,100.00
1,wk_gender,1.000000,100.00
2,wk_course_completed_at,1.000000,100.00
3,user_grade_changed_at,0.999553,99.96
4,wk_officially_started_at,0.998361,99.84
5,enrollment_date,0.557211,55.72
6,status_actual,0.265942,26.59
7,target,0.265942,26.59
8,training_finished_at_last,0.215137,21.51
9,training_mark_max,0.215137,21.51


## Что делает большая сборочная ячейка

Ниже по шагам описано, что именно делает предыдущая ячейка и зачем это нужно.

1. Сначала для разных сущностей восстанавливается `course_id`.
   Для этого используются цепочки `lesson -> course`, `group -> lesson -> course`, `training -> lesson -> course`, а также доступные связи для домашних заданий.
   Это нужно потому, что далеко не все сырые события сразу лежат на уровне курса.

2. Затем собираются признаки структуры самого курса.
   Считается число уроков, задач, тренингов, домашних заданий и элементы структуры курса, которые одинаковы для всех пользователей внутри одного `course_id`.

3. После этого подготавливаются user-level и users-course-level признаки.
   Из `users` берутся базовые характеристики пользователя, а из `users_courses` — признаки состояния прохождения курса, доступов и накопленных баллов.

4. Отдельно агрегируется `user_lessons`.
   Здесь строятся признаки по урокам внутри курса: число уроков, групп, просмотров, решённых уроков, суммарные баллы и т.д.

5. Далее агрегируется `user_activity_histories`.
   События внутри урока сначала связываются с `user_lessons`, затем через урок восстанавливается курс, после чего считаются признаки активности: число событий, число затронутых уроков, число дней активности и распределение по типам действий.

6. Затем агрегируются `wk_users_courses_actions`.
   Это уже course-level лог действий: он даёт число событий по курсу, активные дни, временные границы активности и частоты конкретных action-типов.

7. После этого агрегируются просмотры медиа из `wk_media_view_sessions`.
   Для них восстанавливается курс через урок или группу, а затем считаются признаки глубины просмотра: число сессий, средняя доля просмотра, максимальная доля, число просмотров на 80% и полных просмотров.

8. Следующим шагом агрегируются `user_trainings`.
   Это даёт признаки по тренингам: число прохождений, число решённых задач, баллы, попытки, оценки и временные границы активности.

9. Затем агрегируются `user_answers`.
   Перед этим для ответов восстанавливается `course_id` отдельно для ресурсов типа `Lesson`, `Training` и `Homework`.
   После этого считаются course-level признаки по ответам: число задач, успешных решений, пропусков, partial answers, баллов, попыток и границы по времени.

10. Когда все сырые источники агрегированы, формируется population-таблица.
    Она задаётся через `modules/status_modules_complete.csv`: одна строка на один объект `module + user_id + course_id`.

11. На последнем шаге все агрегаты последовательно присоединяются к population-таблице.
    В результате получается единая витрина на уровне `user-course`, пригодная для дальнейшей работы с моделью.

12. В самом конце дополнительно считаются таблицы покрытия и пропусков.
    Они помогают понять, какие источники действительно хорошо наполняют витрину, а какие дают неполное покрытие и потребуют осторожности на следующем этапе.
    При этом `missingness_df` считается без `module == 4`, потому что для четвёртого модуля поставка сейчас заведомо неполная и она искусственно ухудшает картину пропусков.


# Выгрузка готового датасета

In [25]:

print("Сэмпл итоговой user-course витрины:")
display(modeling_dataset_df.head())
print("Итоговый экспорт:", export_path)
print("Форма итоговой user-course витрины:", modeling_dataset_df.shape)

Сэмпл итоговой user-course витрины:


,module,user_id,course_id,club_name,teacher_id,cohort_id,level_bin,enrollment_date,status_actual,target,users_course_id,state,created_at,updated_at,access_finished_at,wk_points,wk_max_points,wk_max_viewable_lessons,wk_max_task_count,wk_officially_started_at,wk_course_completed_at,course_points_ratio,user_created_at,user_updated_at,user_type,sign_in_count,user_last_sign_in_at,grade_id,subscribed,is_teacher,timezone,user_grade_changed_at,xp,d_wk_school_id,d_wk_municipal_id,d_wk_region_id,wk_gender,course_lessons_total,course_conspect_expected_count,course_task_expected_count,course_survival_training_expected_count,course_attendance_tracking_enabled_count,course_video_duration_total,course_video_duration_mean,course_lesson_tasks_total,course_lesson_tasks_unique,course_required_tasks_total,course_trainings_total,course_training_tasks_total,course_homeworks_total,course_homework_items_total,access_periods_count,access_days_total,access_started_at_first,access_expired_at_last,user_lessons_rows,user_lessons_nunique,lessons_logged_nunique,groups_logged_nunique,video_visited_lessons,video_viewed_lessons,translation_visited_lessons,solved_lessons_count,solved_tasks_total,user_lessons_points_sum,wk_solved_task_count_total,video_viewed_share_by_logged_lessons,lesson_activity_events_total,lesson_activity_user_lessons_nunique,lesson_activity_days,lesson_activity_first_at,lesson_activity_last_at,lesson_activity_count__show_conspect,lesson_activity_count__visit_translation,lesson_activity_count__visit_video,actions_total,action_days,actions_with_lesson_id,action_first_at,action_last_at,action_count__scratch_playground_visited,action_count__start_training,action_count__user_answer,action_count__visit_preparation_material,action_count__visit_translation,action_count__visit_video,media_sessions_total,media_resources_nunique,media_watch_share_mean,media_watch_share_max,media_sessions_80pct,media_sessions_full,media_started_at_first,media_started_at_last,trainings_rows,trainings_unique_total,training_solved_tasks_total,training_earned_points_sum,training_submitted_answers_sum,training_attempts_sum,training_mark_max,training_started_at_first,training_finished_at_last,answer_rows,answer_tasks_nunique,answer_solved_count,answer_skipped_count,answer_partial_count,answer_points_sum,answer_attempts_sum,answer_max_attempts_sum,answer_async_nonzero_count,answer_first_submitted_at,answer_last_submitted_at
0,1,701741,1029,Python. Начальный уровень. Онлайн. 1 модуль,699869,1149,0,2025-09-19,Завершил,1,588595,active,2025-10-28 17:22:00,2025-12-05 12:20:00,2026-04-28,138.0,140.0,20.0,153.0,NaT,NaT,0.985714,2025-09-08 02:41:00,2026-03-27 09:25:00,User::Pupil,13,2026-03-25 08:24:00,3010,True,False,Asia/Krasnoyarsk,NaT,649,285701,238671,238670,<NA>,23,20,20,False,0,0.0,<NA>,110,110.0,110.0,3.0,43,20.0,60,1,182.0,2025-10-28,2026-04-28,23,23.0,23.0,23.0,20.0,13.0,0.0,23.0,153,138.0,153,0.565217,24.0,20.0,1.0,2025-12-05 10:37:00,2025-12-05 11:35:00,0.0,0.0,24.0,51.0,1.0,0,2025-12-05 10:36:00,2025-12-05 12:16:00,0.0,3.0,35.0,0.0,0.0,13.0,23,20.0,0.869565,1.0,20.0,17.0,2025-12-05 07:38:00,2025-12-05 08:34:00,3,3.0,43,30.0,43,3,5.0,2025-12-05 11:10:00,2025-12-05 12:20:00,117,117.0,116.0,0.0,0.0,116.0,117,117,9,2025-12-05 10:36:00,2025-12-05 12:19:00
1,1,737977,1029,Python. Начальный уровень. Онлайн. 1 модуль,730341,1216,0,2025-11-05,Завершил,1,623160,active,2025-11-13 16:40:00,2025-11-29 13:54:00,2026-05-13,118.5,140.0,20.0,153.0,NaT,NaT,0.846429,2025-11-12 10:16:00,2026-03-25 10:14:00,User::Pupil,59,2026-03-24 21:14:00,3010,True,False,Asia/Yekaterinburg,NaT,1094,285701,238671,238670,<NA>,23,20,20,False,0,0.0,<NA>,110,110.0,110.0,3.0,43,20.0,60,1,181.0,2025-11-13,2026-05-13,23,23.0,23.0,23.0,13.0,10.0,19.0,23.0,153,118.5,153,0.434783,67.0,20.0,12.0,2025-11-14 12:27:00,2025-11-29 13:31:00,0.0,39.0,28.0,123.0,12.0,0,2025-11-14 12:27:00,2025-11-29 13:32:00,0.0,3.0,58.0,0.0,39.0,23.0,30,20.0,0.652770,1.0,14.0,9.0,2025-11-14 09:28:00,2025-11-28 01:29:00,3,3.0,43,27.0,43,3,5

Итоговый экспорт: /Users/maria/Desktop/uni/hackathon/artifacts/user_course_modeling_base.csv
Форма итоговой user-course витрины: (8401, 114)


## Итоги и основные выводы

### Что нашли

- Для задачи предсказания статуса правильная гранулярность — `user-course`, а не просто `user`.
- Не все сырые таблицы одинаково хорошо связаны напрямую с курсом, поэтому часть признаков приходится восстанавливать через уроки, группы, тренинги и типы ресурсов.
- `modules/status_modules_complete.csv` хорошо задаёт population по всем четырём модулям, но не всегда напрямую стыкуется с `users_courses`, поэтому для итоговой витрины нужен смешанный способ обогащения.

### Что поправили

- При загрузке данных учтён разделитель тысяч `,` внутри числовых значений.
- Таблицы приведены к типам, удобным для анализа: даты распарсены, числовые поля приведены к nullable-типам, категориальные поля выделены отдельно.
- Перед построением итоговой витрины из событийных таблиц удаляются точные дубли, чтобы не завышать счётчики активности, просмотров, ответов и периодов доступа.
- Для документа собраны проверки по дубликатам, ключам, связности, базовым временным аномалиям и покрытию признаков по модулям.

### Что получили

- Сформирована единая user-course population-таблица на базе `modules/status_modules_complete.csv`.
- К ней присоединены признаки структуры курса, профиля пользователя, доступов, уроков, событий внутри урока, course-level действий, просмотров медиа, тренингов и ответов.
- В результате получена витрина, где одна строка соответствует одной паре `user-course`.
- Итоговый файл сохраняется в `./artifacts/user_course_modeling_base.csv` и может использоваться как база для следующего этапа: анализа пропусков, отбора признаков, постановки temporal cutoff/masking и последующего обучения модели.
- При анализе пропусков `module == 4` исключается из `missingness_df`, потому что его текущая поставка неполная и без такого исключения таблица пропусков становится менее интерпретируемой.


# ВАЖНО!

На текущий момент в датасете присутсвтуют все данные, которые у нас есть. Для целей обучения модели следующими итерациями надо будет пересобрать датасет со всеми данными, которые доступны на момент середины каждого моудуля (Это условно, отдельная задача - выбрать, на какой момент с начала модуля модель должна предсказывать результаты), чтобы избежать data leakage.